# Lesson 4.5: Merging, Cleaning, and Reshaping

**Analyst:** Meron Welderufael
**Date:** 11/4/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Clean the marketing team's messy plan_metadata file,
merge it into the customer data, and produce the consolidated per-tier
summary table for Hannah's 2 PM checkpoint.

Friday checkpoint: 3 PM. Memo table to Hannah by 2 PM for review.

Load the customer data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('finflow-customers-200k.csv',
                parse_dates = ['signup_date', 'last_login_date', 'churn_date'])

print(f'Customer Data: {df.shape}')
df.head(2)

Customer Data: (200000, 17)


,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date,referral_source
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2026-03-11,Basic,9.99,2026-10-14,91,11.80,4,True,NaT,Organic Search
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-12-06,Basic,9.99,2026-10-05,328,3.30,1,True,NaT,Social Media


## Inspecting the marketing team's plan metadata

Load it. Look at it. Document every problem before fixing anything.

In [2]:
plans_raw = pd.read_csv('plan_metadata_raw.csv',
                       parse_dates=['launch_date'])
print(f'Shape: {plans_raw.shape}')
print()
plans_raw

Shape: (5, 5)



,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date
0,Basic,Starter Plan,Individual,9.99,2019-03-15
1,PRO,Professional,SMB,29.99,2020-08-01
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30
3,PRO,Professional,SMB,29.99,2020-08-01
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12


In [8]:
# Survey the problems
print("Unique plan_tier values")
for value in plans_raw['plan_tier'].unique():
    print(f" '{value}' (length {len(value)})")
print()

#Missing Values
print('Missing values per column:')
print(plans_raw.isna().sum())
print()

#Duplicates
print(f"Duplicate rows: {plans_raw.duplicated().sum()}")

Unique plan_tier values
 ' Basic ' (length 7)
 'PRO' (length 3)
 ' enterprise ' (length 12)
 'Legacy' (length 6)

Missing values per column:
plan_tier             0
marketing_name        0
target_segment        0
monthly_fee_listed    1
launch_date           0
dtype: int64

Duplicate rows: 1


**Observation:** Five rows total, four distinct after deduplication.
Plan tier values have whitespace and capitalization mess that will prevent
the join column from matching the customer data. monthly_fee_listed is
missing for Legacy. Need to clean before merging.

In [10]:
# Strip whitespace, then title-case
plans_raw['plan_tier_clean']=plans_raw['plan_tier'].str.strip().str.title()

print("Cleaned plan_tier values")
for values in plans_raw['plan_tier_clean'].unique():
    print(f"{values}: (length {len(values)})")


Cleaned plan_tier values
Basic: (length 5)
Pro: (length 3)
Enterprise: (length 10)
Legacy: (length 6)


In [15]:
# How many rows before
print(f"Rows before dedupe: {len(plans_raw)}")

# Drop duplicates on the cleaned column
plans = plans_raw.drop_duplicates(subset=['plan_tier_clean'], keep='first').copy()

print(f"Rows after dedupe: {len(plans)}")
plans

Rows before dedupe: 5
Rows after dedupe: 4


,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date,plan_tier_clean
0,Basic,Starter Plan,Individual,9.99,2019-03-15,Basic
1,PRO,Professional,SMB,29.99,2020-08-01,Pro
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30,Enterprise
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12,Legacy


In [18]:
# Confirm the NaN remains

In [21]:
print('monthly_fee_listed values')
print(plans[['plan_tier', 'monthly_fee_listed']])
plans

monthly_fee_listed values
      plan_tier  monthly_fee_listed
0        Basic                 9.99
1           PRO               29.99
2   enterprise               199.00
4        Legacy                 NaN


,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date,plan_tier_clean
0,Basic,Starter Plan,Individual,9.99,2019-03-15,Basic
1,PRO,Professional,SMB,29.99,2020-08-01,Pro
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30,Enterprise
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12,Legacy


**Observation:** Leaving the Legacy monthly_fee_listed as NaN rather than
filling. The customer data already has a per-customer monthly_fee, so the
listed price isn't critical for the join. If Hannah asks during the
readout, we can investigate.

In [20]:
print("Customer plan_tier values:")
print(df['plan_tier'].value_counts())

Customer plan_tier values:
plan_tier
Basic         120016
Pro            59878
Enterprise     20106
Name: count, dtype: int64


In [25]:
# Inner join: only rows where plan_tier_clean matches on both sides
df_inner = pd.merge(
    df,
    plans,
    left_on= 'plan_tier',
    right_on= 'plan_tier_clean',
    how = 'inner'
)
print(f'Customer rows before merge: {len(df)}')
print(f'customer rows after merge: {len(df_inner)}')
print(f'columns added: {df_inner.shape[1] - df.shape[1]}')
print()
print('Tier counts in inner merge result:')
print(df_inner['plan_tier_clean'].value_counts())

Customer rows before merge: 200000
customer rows after merge: 200000
columns added: 6

Tier counts in inner merge result:
plan_tier_clean
Basic         120016
Pro            59878
Enterprise     20106
Name: count, dtype: int64


In [28]:
df_enriched = pd.merge(
    df,
    plans,
    left_on='plan_tier',
    right_on= 'plan_tier_clean',
    how = 'left'
)
print(f'shape after merge: {df_enriched.shape}')
print(f"NaN counts in metadata columns:")
print(df_enriched[['marketing_name', 'target_segment', 'monthly_fee_listed', 'launch_date']].isna().sum())

shape after merge: (200000, 23)
NaN counts in metadata columns:
marketing_name        0
target_segment        0
monthly_fee_listed    0
launch_date           0
dtype: int64


In [33]:
df_outer=pd.merge(
    df[['customer_id', 'plan_tier']],
    plans,
    left_on='plan_tier',
    right_on= 'plan_tier_clean',
    how = 'outer'
)
print(f"Outer merge shape: {df_outer.shape}")
print()
print("Rows where customer_id is NaN (these are unmatched metadata rows):")
print(df_outer[df_outer['customer_id'].isna()])

Outer merge shape: (200001, 8)

Rows where customer_id is NaN (these are unmatched metadata rows):
       customer_id plan_tier_x plan_tier_y marketing_name target_segment  \
140122         NaN         NaN      Legacy  Heritage Plan     Individual   

        monthly_fee_listed launch_date plan_tier_clean  
140122                 NaN  2017-06-12          Legacy  


**Observation:** Outer join exposed the Legacy plan, which exists in
metadata but not in customer data. If Hannah cares about Legacy customers
(she might; it could be a deprecated tier with no current subscribers),
flag in the memo.

In [42]:
# Re-create the per-tier summary, now with marketing names
def churn_rate_pct(active_series):
    return (~active_series).mean()*100

memo_summary = df_enriched.groupby(['plan_tier_clean', 'marketing_name', 'target_segment']).agg({
    'customer_id': 'count',
    'monthly_fee': 'sum',
    'active': churn_rate_pct,
    'total_sessions': 'mean'
})

memo_summary.columns = ['customer_count', 'total_mrr', 'churn_rate_pct', 'avg_sessions']
memo_summary = memo_summary.reset_index()
memo_summary

,plan_tier_clean,marketing_name,target_segment,customer_count,total_mrr,churn_rate_pct,avg_sessions
0,Basic,Starter Plan,Individual,120016,1198959.84,25.08,163.94
1,Enterprise,Enterprise Plus,Enterprise,20106,4001094.00,4.96,201.22
2,Pro,Professional,SMB,59878,1795741.22,14.86,182.60


In [44]:
sessions_by_tier_and_status = df_enriched.pivot_table(
    index = 'marketing_name',
    columns= 'active',
    values = 'total_sessions',
    aggfunc = 'mean'
)
sessions_by_tier_and_status.columns = ['Churned_avg_Sessions', 'active_avg_sessions']
sessions_by_tier_and_status

,Churned_avg_Sessions,active_avg_sessions
marketing_name,,
Enterprise Plus,25.53,210.40
Professional,25.63,210.00
Starter Plan,25.41,210.32


In [45]:
memo_summary.to_csv('friday-memo-summary.csv', index=False)
sessions_by_tier_and_status.to_csv('friday-memo-sessions-comparison.csv')
print('saved both files.')

saved both files.


In [47]:
mrr_by_tier = df_enriched.groupby('marketing_name')['monthly_fee'].sum()
mrr_pct = (mrr_by_tier / mrr_by_tier.sum() * 100).round(1)
print("MRR percent by marketing tier:")
print(mrr_pct)



MRR percent by marketing tier:
marketing_name
Enterprise Plus   57.20
Professional      25.70
Starter Plan      17.10
Name: monthly_fee, dtype: float64


**AI Use Note:** Used Claude to confirm the per-group-percent idiom.
Output matched the obvious groupby pattern. Verified on FinFlow data.
Tier 1 disclosure.